# Lab 2 · Ingest an external **API** into Bronze

Pull live regional air quality (PSI) from Singapore's open data API and land it in bronze.
This gives Rahim's activity *environmental context* — hazy days explain skipped outdoor events.

> **Attach** the `lh_resident360` Lakehouse first.

In [ ]:
import requests, datetime
from pyspark.sql import Row, functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

## 1. Call the data.gov.sg PSI API

In [ ]:
url = "https://api.data.gov.sg/v1/environment/psi"
try:
    resp = requests.get(url, timeout=30).json()
    items = resp["items"][0]["readings"]["psi_twenty_four_hourly"]
    ts = resp["items"][0]["timestamp"]
    src = 'live API'
except Exception as e:
    # Offline fallback so the lab always runs
    print("API unreachable, using fallback values:", str(e)[:120])
    items = {'west': 55, 'east': 48, 'central': 52, 'north': 60, 'south': 50}
    ts = datetime.datetime.now().isoformat()
    src = 'fallback'
print("source:", src, "| timestamp:", ts)

## 2. Map API regions → our five HPB regions and write bronze

In [ ]:
region_map = {"west":"West","east":"East","central":"Central","north":"North","south":"North-East"}
rows = [Row(reading_ts=ts, region_api=k, psi_24h=int(v), region=region_map.get(k, k.title()))
        for k, v in items.items()]
air = spark.createDataFrame(rows)
air.write.mode("overwrite").saveAsTable("bronze.env_air_quality")
print("bronze.env_air_quality:", spark.table("bronze.env_air_quality").count(), "rows")
display(air)